# TradeSense · Component 03 · notebook 48: does the Autoencoder catch a family the supervised model has never seen? (protocol U1 v2, Part B, leave-one-family-out; CPU)

**Question.** Part A (notebook 47) only showed that AE and XGBoost catch different events. The real test of "previously unseen abnormality" is: take one known family away from XGBoost's training, then ask whether adding the label-free AE (A6) recovers events of that family.

**What this notebook does.** For each of the six known families f it trains a *study copy* of XGBoost without f (same recipe as the frozen v2; the frozen models are only read, never replaced), then compares on family f: XGBoost-without-f (`S_wo`), the AE (`N`), and "either alarms" (`S_wo or N`), on VAL-3 (design, no decision) and CAL-3 (decision, reported once). Hypotheses H-U4 and H-U5 are fixed below before any model exists. TEST data is never loaded. Training takes some minutes per model (18 models); if the time guard stops the run, run all again and finished models are kept.

**Protocol text hashed by this notebook (approved; sha256 must start c448d8d48815aab0):**


```
# Protocol U1 v2 (approved by the user 2026-10-16; frozen when notebook 47 hashes this exact text) - Known + previously unseen abnormality: the "UNSEEN PATTERN" signal of the Autoencoder
Written BEFORE any result of this study exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) is changed. XGBoost, Transformer, AE, fusion, Integrity Score and states stay exactly as they are. The signal below is an ADDITIONAL output.

## Idea (the user's novelty 2)
XGBoost and Transformer recognise KNOWN abnormal patterns. The Autoencoder, trained on normal behaviour only (label-free), flags behaviour it has not seen. When the AE flags a minute while XGBoost and Transformer do not, the dashboard shows a badge "UNSEEN PATTERN" next to the normal state.

## The signal (rule declared now)
UNSEEN PATTERN at bar t = the AE (A6, mean10, N1) is above its own T1 rolling alert line at budget 2/day AND XGBoost and Transformer are both below their own T1 lines (budget 2/day each). Source: the `agree` bit mask of the engine (AE only = 1). No new threshold, no new fitting. Wording on the screen: "Only the label-free Autoencoder flagged this; the trained models did not recognise it. It is a statistical signal, not proof of a new manipulation type."

## Part A - signal on saved TEST-3 scores (no training)
New pre-registered analysis on saved TEST-3 scores (TEST-3 was opened once in T0; nothing is retuned; library sha must be 3b5d9df23521adf9). Per family: three-way split of detected events (AE only / supervised only / both / neither), genuine TEST-B episodes/day of the signal, Sep vs Oct, per symbol.
- H-U1: genuine TEST-B burden of the UNSEEN PATTERN signal (episodes/day) is <= 1.0.
- H-U2 (supervised dominates on known families): on the six known families, recall(XGBoost or Transformer at 2/day) - recall(AE at 2/day) >= 0.20.
- H-U3 (genuinely different catches): macro over all 8 families, at least 15% of the events caught by the AE are missed by fusion.

## Part B - the real "unseen" test: leave-one-family-out (LOFO) system study (VAL-3 for design, CAL-3 reported once)
For each of the six known families f: train a STUDY copy of XGBoost (same config as the frozen v2: A | pooled | depth 3, seeds 11/22/33, N1) on TRAIN-3 WITHOUT family f; the frozen models are untouched and not replaced. Evaluate on family f (T1c, 2/day, mean BTC/ETH, same windows and episodes as notebook 29b): S_wo = XGBoost without f; N = AE (A6); "S_wo or N" = event detected if either alarms; genuine burden of the OR reported.
- H-U4 (the AE adds on a truly unseen family): macro over the six families, recall(S_wo or N) - recall(S_wo) >= 0.05 with paired event-bootstrap 95% interval above 0, on CAL-3.
- H-U5 (burden): genuine episodes/day of "S_wo or N" <= S_wo's + 2.1 (the AE's own budget plus its observed rate).
- Descriptive per family. Transformer LOFO (GPU) is a later extension, not part of this protocol version.

## Part C (conditional, only if H-U4 fails) - AE v3 as a NEW version
New folder, A6 AE kept; selection by family-level cross-validation on VAL-3; test on a freshly injected set with new seeds on the TEST-B genuine bars ("TEST-3b"), declared before it is built. Separate protocol.

## Part D (separate, after the data format is checked): real external evidence
Layer 2 (Sapienza pump-and-dump, Mt.Gox) with the domain shift stated; claimed only if the frozen features can be built from that data.

## Known weaknesses, stated in advance
AE alone is weak (38.2% macro-8 at 2/day in TEST-3), so H-U3 or H-U4 may fail and will be reported as failed. The two "new" families (momentum_ignition, slow_accumulation) share mechanics with trained families, so they are not a clean unseen test, which is why Part B exists. Labels are synthetic mechanisms. LOFO with XGBoost only first.

## Operational readings written before running notebook 47 (Part A)
- Signal bar = AE alarm bar AND NOT XGBoost alarm bar AND NOT Transformer alarm bar, each by the T1c rule (rolling genuine-series threshold, q frozen per budget 2 and symbol). Episodes merge bars within 5 minutes.
- "XGBoost or Transformer" = an event is detected if XGBoost (own 2/day rule) or Transformer (own 2/day rule) alarms inside the event window. "Caught by the AE" = the AE (2/day rule) alarms inside the event window; "missed by fusion" = fusion (2/day) does not.
- H-U3: for each family, share_f = (events caught by the AE and missed by fusion) / (events caught by the AE), pooled over both symbols; macro = mean over the families with at least 5 events caught by the AE (the others are listed, not averaged).
- Descriptive only: budgets 1 and 4 (the AE budget 0.5 does not exist in declared_ae_v2.json and is dropped), per symbol, Sep versus Oct, chance level of the signal (alarms rolled in time).

```

In [ ]:
import os, sys, json, time, hashlib, warnings, glob, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; XG = OUT/'xgb'; XG48 = OUT/'xgb_lofo_study'; XG48.mkdir(parents=True, exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]; MODE = 'N1'; CFG = ('A', 'pooled', 3)
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150)); T_START = time.time(); NTHREAD = max(1, os.cpu_count() or 1)
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
def out_of_time(): return (time.time() - T_START) / 60 > MAX_RUN_MINUTES
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
pu = OUT/'protocol_u1_v2.md'; assert pu.exists(), 'run notebook 47 first (it hashes protocol U1 v2)'
PHU = hashlib.sha256(pu.read_text().encode()).hexdigest(); assert PHU.startswith('c448d8d48815aab0'), 'protocol U1 v2 differs from the approved text'
assert PHU == (OUT/'protocol_u1_v2.sha256').read_text().split()[0]
READ48 = r"""Protocol U1 v2, Part B - operational readings (notebook 48), written BEFORE any LOFO model exists. They add no new freedom: they only fix words of the approved text.
1. Study copies: for each held-out family f in {1..6} an XGBoost trained like the frozen v2 (feature set A = 53 features, pooled BTC+ETH, depth 3, normalisation N1, same parameters, 800 rounds max, early stopping 40 on VAL-3, seeds 11/22/33, scores averaged as margins). Positive rows of family f are removed from the fit AND from the early-stopping rows (VAL-3 keeps the other five old families only). The two new families (7, 8) are never used. Frozen models are only read.
2. S_wo = that study model. N = the declared AE (A6: mean10 of AE-ctx fmax, N1; scores/ae_ctx_agg_*_N1.npz column 0). Alarm rule for both = T1c: rolling 7-day quantile of the model's own score on the GENUINE series, applied to the injected series; q = smallest grid value with <= 2 episodes/day on the genuine DEV bars (90,720..129,599), per symbol and per study model. For the AE the frozen q of declared_ae_v2.json (budget 2) is used as declared; a recomputation from the saved scores is only printed. Episodes merge bars within 5 minutes. 'S_wo or N' = alarm bar of either.
3. Event detected = an alarm bar inside [t0, t0+dur-1]. Recall of family f = hits/events within symbol, then mean of BTC and ETH. Macro = mean over the six families, each family scored by ITS OWN study model.
4. H-U4 is decided on CAL-3 (reference genuine series = August bars, events 2024-08-02..08-31, scores sliced to the first 31 days). Gain = macro recall(S_wo or N) - macro recall(S_wo). Paired event bootstrap: events resampled within symbol x family (same indices for both arms), 2000 draws, seed 0, 95% percentile interval; H-U4 holds if gain >= 0.05 and the interval's lower end > 0. VAL-3 is the design set: reported, no decision.
5. H-U5 is decided on the CAL-3 genuine August bars (bars >= 1,440): mean over the six families and both symbols of the genuine episodes/day of 'S_wo or N' <= mean of S_wo's own + 2.1. The worst family and the DEV values are listed, not decided.
6. CAL-3 numbers are produced once by this notebook and are not used to change anything. If H-U4 fails, Part C is opened as a separate protocol; nothing here is retuned.
"""
RH = hashlib.sha256(READ48.encode()).hexdigest(); rp = OUT/'protocol_u1_v2_partB_readings.md'
if rp.exists(): assert rp.read_text() == READ48, 'operational readings exist and differ from this notebook'
else: rp.write_text(READ48); (OUT/'protocol_u1_v2_partB_readings.sha256').write_text(RH + '\n')
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
assert sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'b2_lib differs from the one that opened TEST'
for f_ in ['declared_ae_v2.json', 'xgb_selection_table.csv', 'xgb_val_table_T1c.csv']: assert (OUT/f_).exists(), f_ + ' missing'
DAE = json.loads((OUT/'declared_ae_v2.json').read_text()); assert DAE['variant'] == 'mean10' and DAE['normalisation'] == 'N1'
for s in SYMBOLS: assert (OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz').exists() and (OUT/'scores'/f'ae_ctx_{s}_N1.npz').exists()
sys.path.insert(0, str(OUT)); import b2_lib as B
say(f'protocol U1 v2 sha256 {PHU[:16]}; Part B readings sha256 {RH[:16]}; protocol B2 intact {ph[:16]}; xgboost {xgb.__version__}; {NTHREAD} threads' + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol U1 v2 sha256 c448d8d48815aab0; Part B readings sha256 9da1844cd7f2f20b; protocol B2 intact 443f99d5faf5929f; xgboost 3.4.1; 2 threads


## 1 · Data, features, AE scores

In [ ]:
LOOK = 15; FEATS = B.FEATS; PB = B.PRIMARY; OLD = [1, 2, 3, 4, 5, 6]; AOLEN = 31 * 1440; CAL_LO = 1440
L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
D = {}; SCO = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet').iloc[:AOLEN]      # genuine August only: TEST-B is never loaded
    for d in (a, b): d.loc[d.index[:15], 'volume_acceleration'] = 0.0
    sets = {}
    for name in ['TRAIN', 'VAL', 'CAL']:
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); ev = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
        sets[name] = {int(r): (g.reset_index(drop=True), ev[ev.replica == r].reset_index(drop=True)) for r, g in f.groupby('replica')}
    D[s] = {'GEN_L1': a[FEATS].reset_index(drop=True), 'GEN_AO': b[FEATS].reset_index(drop=True), 'sets': sets}
    z = np.load(OUT/'scores'/f'ae_ctx_{s}_{MODE}.npz'); SCO[(s, MODE)] = {k: z[k] for k in z.files if k != 'scale'}
    for k, v in SCO[(s, MODE)].items():
        n_ = len(D[s]['GEN_L1']) if k == 'GEN_L1' else len(D[s]['GEN_AO']) if k == 'GEN_AO' else len(D[s]['sets'][k.split('_r')[0]][int(k.split('_r')[1])][0])
        assert len(v) == n_ and v.shape[1] == 8, (s, k)
    say(f'{s}: genuine Layer 1 {len(a)} bars, genuine Aug {len(b)} bars; replicas ' + str({k: len(v) for k, v in sets.items()}))
assert not any('TEST' in k for s in SYMBOLS for k in SCO[(s, MODE)])
N0 = {s: B.fit_n0(D[s]['GEN_L1'].iloc[15:90720]) for s in SYMBOLS}

def build_feats(Z):
    """53 causal rolling features of the six normalised features (identical recipe to XGBoost v1/v2, look-back <= 15 bars)."""
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:B.WARM] = np.nan; return out, list(cols)
NAMES_A = build_feats(np.zeros((200, 6)))[1]; NA = len(NAMES_A); assert NA == 53
XS = {}
def series_F(s, key):
    if key in ('GEN_L1', 'GEN_AO'): return D[s][key]
    name, r = key.split('_r'); return D[s]['sets'][name][int(r)][0]
def XA(s, key):
    """feature set A (53 columns) of a series under N1; CAL / genuine-August matrices are not cached."""
    k = (s, key)
    if k in XS: return XS[k]
    z = B.normalise(series_F(s, key), MODE, N0[s]); X = build_feats(z)[0]
    if not key.startswith(('CAL', 'GEN_AO')): XS[k] = X
    return X
RS = {}
for s in SYMBOLS:
    for name in ['TRAIN', 'VAL']:
        for r, (f, ev) in D[s]['sets'][name].items():
            lab = f.label.to_numpy().astype(int); inseg = f.in_seg.to_numpy().astype(bool); n = len(f); idx = np.arange(n)
            c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
            clean = ((c[idx + 1] - c[np.maximum(idx - LOOK, 0)]) == 0) & inseg & (idx >= B.WARM)
            pos = (lab > 0) & inseg & (idx >= B.WARM); assert not (clean & pos).any(); RS[(s, name, r)] = (lab, inseg, clean, pos)
# ---- events, AE scores (A6 = mean10, column 0 of the aggregated files)
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
ES = {}
for name in ['VAL', 'CAL']:
    ES[name] = [(s, f'{name}_r{r}', D[s]['sets'][name][r][1]) for s in SYMBOLS for r in sorted(D[s]['sets'][name])]
META = {n: pd.concat([pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy()}) for s, k, ev in ES[n]], ignore_index=True) for n in ES}
for s, k, ev in ES['CAL']: assert ev.t0.min() >= CAL_LO and (ev.t0 + ev.dur).max() <= AOLEN, 'CAL-3 events must lie inside 2024-08-02..08-31'
for s, k, ev in ES['VAL']: assert ev.t0.min() >= 90720 and (ev.t0 + ev.dur).max() <= 129600
AE = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz'); assert not any('TEST' in k for k in z.files)
    for k in z.files:
        if k == 'scale' or k.startswith('TRAIN'): continue
        a = z[k][:, 0]; a = a[:AOLEN] if k.startswith('CAL') else a; AE[(s, k)] = nanw(a)
say('events per set: ' + str({n: META[n].groupby('sym').size().to_dict() for n in META}) + '; CAL-3 per family (both symbols): ' + str(META['CAL'].fam.value_counts().sort_index().to_dict()))
# ---- the selection recorded by notebook 29 (must be the config this protocol studies)
TAB = pd.read_csv(OUT/'xgb_selection_table.csv'); tm = TAB[TAB['mode'] == MODE]
bestA = tm[tm.fs == 'A'].metric.max(); bestB = tm[tm.fs == 'B'].metric.max(); al_ = 'B' if bestB - bestA >= 0.03 else 'A'; tt = tm[tm.fs == al_]; tied = tt[tt.metric >= tt.metric.max() - 0.05].copy()
tied['k'] = list(zip((tied.scope != 'pooled').astype(int), tied.depth, -tied.pr_auc)); FROZEN = tied.sort_values('k').iloc[0].config
if not MOCK: assert FROZEN == 'A|pooled|d3', f'frozen N1 selection is {FROZEN}, protocol U1 studies A|pooled|d3'
else: say(f'[MOCK] frozen selection in the mock is {FROZEN}; the study still uses A|pooled|d3')
say('data ready; XGBoost config studied: A | pooled | depth 3, N1; frozen selection recorded by notebook 29: ' + FROZEN)

BTCUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars; replicas {'TRAIN': 2, 'VAL': 2, 'CAL': 1}
ETHUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars; replicas {'TRAIN': 2, 'VAL': 2, 'CAL': 1}
events per set: {'VAL': {'BTCUSDT': 300, 'ETHUSDT': 300}, 'CAL': {'BTCUSDT': 200, 'ETHUSDT': 200}}; CAL-3 per family (both symbols): {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50}
data ready; XGBoost config studied: A | pooled | depth 3, N1; frozen selection recorded by notebook 29: A|pooled|d3


## 2 · Study models: XGBoost without one family (18 models)

In [ ]:
PARAMS = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'scale_pos_weight': 5.0, 'nthread': NTHREAD}
MAX_ROUNDS, ES_ROUNDS = 800, 40
ROWLOG = []
def get_rows(s, name, lofo, tag):
    """positives from every replica; clean negatives from replica 0 only. VAL: only the 6 old families; the held-out family removed from both."""
    Xs, ys = [], []
    for r, (f, ev) in D[s]['sets'][name].items():
        lab, inseg, clean, pos = RS[(s, name, r)]; p = pos.copy()
        if name == 'VAL': p &= (lab <= 6)
        p &= (lab != lofo)
        m = p | (clean if r == 0 else np.zeros(len(lab), bool)); ii = np.flatnonzero(m)
        ROWLOG.append({'tag': tag, 'sym': s, 'set': name, 'min_bar': int(ii.min()), 'max_bar': int(ii.max()), 'n': int(len(ii)), 'n_pos': int(p[ii].sum()), 'pos_labels': sorted(set(lab[ii][p[ii]].tolist())), 'outside_segment': int((~inseg[ii]).sum())})
        Xs.append(XA(s, f'{name}_r{r}')[ii]); ys.append(p[ii].astype(np.float32))
    return np.concatenate(Xs), np.concatenate(ys)
def spath(fam, seed): return XG48/f'N1_A_pooled_pool_d3_s{seed}_no{fam}.json'
def train_study(fam, seed):
    p = spath(fam, seed); ip = Path(str(p)[:-5] + '.info.json')
    if p.exists() and ip.exists(): return False
    tag = f'study|no{fam}|s{seed}'; n0_ = len(ROWLOG)
    parts = [get_rows(s, 'TRAIN', fam, tag + '|fit') for s in SYMBOLS]; ves = [get_rows(s, 'VAL', fam, tag + '|es') for s in SYMBOLS]
    Xt = np.concatenate([x[0] for x in parts]); yt = np.concatenate([x[1] for x in parts]); Xv = np.concatenate([x[0] for x in ves]); yv = np.concatenate([x[1] for x in ves])
    dtr = xgb.DMatrix(Xt, label=yt, feature_names=NAMES_A); dva = xgb.DMatrix(Xv, label=yv, feature_names=NAMES_A); t = time.time()
    bst = xgb.train({**PARAMS, 'max_depth': 3, 'seed': seed}, dtr, num_boost_round=MAX_ROUNDS, evals=[(dva, 'val')], early_stopping_rounds=ES_ROUNDS, verbose_eval=False)
    bst.save_model(str(p)); info = {'family_removed': fam, 'best_iteration': int(bst.best_iteration), 'best_aucpr_val': float(bst.best_score), 'n_fit': int(len(yt)), 'n_fit_pos': int(yt.sum()), 'n_es': int(len(yv)), 'n_es_pos': int(yv.sum()), 'seconds': round(time.time() - t, 1), 'rows': ROWLOG[n0_:]}
    ip.write_text(json.dumps(info)); say(f'  trained study model without family {fam} ({B.FAMS[fam]}), seed {seed}: best iteration {info["best_iteration"]}, fit {len(yt):,} rows ({int(yt.sum()):,} positive), {info["seconds"]}s'); return True
class SM:
    """mean-margin ensemble of boosters from a list of files"""
    def __init__(self, paths): self.bst = [(lambda b_, p: (b_.load_model(str(p)), b_)[1])(xgb.Booster(), p) for p in paths]; self.its = [json.loads(Path(str(p)[:-5] + '.info.json').read_text())['best_iteration'] for p in paths]; self.cache = {}
    def score(self, s, key):
        if (s, key) not in self.cache:
            dm = xgb.DMatrix(XA(s, key), feature_names=NAMES_A); m = np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in zip(self.bst, self.its)], axis=0).astype(np.float32).astype('float64'); m[:B.WARM] = np.nan      # float32 as in notebook 29b
            if key.startswith('CAL'): m = m[:AOLEN]
            self.cache[(s, key)] = m
        return self.cache[(s, key)]
TRAIN_DONE = True
say('--- training 6 families x 3 seeds of study models (A | pooled | depth 3, N1) ---')
for fam in OLD:
    for sd in SEEDS:
        if out_of_time(): TRAIN_DONE = False; break
        train_study(fam, sd)
if not TRAIN_DONE: say('>>> time budget reached: Run all again to continue (finished models are kept); the evaluation below is skipped until all 18 models exist. <<<')
else:
    ok = True
    for p in XG48.glob('*.info.json'):
        inf = json.loads(p.read_text())
        for r in inf['rows']:
            ok &= r['outside_segment'] == 0 and inf['family_removed'] not in r['pos_labels'] and not any(l > 6 for l in r['pos_labels'])
            if r['tag'].endswith('|fit'): ok &= r['max_bar'] < 90720 and r['min_bar'] >= B.WARM
            if r['tag'].endswith('|es'): ok &= r['min_bar'] >= 90720 and r['max_bar'] < 129600
    say(('PASS' if ok else 'FAIL') + ' T01 fit rows inside TRAIN-3, early-stopping rows inside VAL-3, held-out family and the two new families never used'); assert ok

--- training 6 families x 3 seeds of study models (A | pooled | depth 3, N1) ---
  trained study model without family 1 (pump_and_dump_like), seed 11: best iteration 796, fit 171,686 rows (20,050 positive), 31.3s
  trained study model without family 1 (pump_and_dump_like), seed 22: best iteration 799, fit 171,686 rows (20,050 positive), 31.2s
  trained study model without family 1 (pump_and_dump_like), seed 33: best iteration 799, fit 171,686 rows (20,050 positive), 31.3s
  trained study model without family 2 (fake_breakout_like), seed 11: best iteration 799, fit 172,888 rows (21,252 positive), 33.9s
  trained study model without family 2 (fake_breakout_like), seed 22: best iteration 798, fit 172,888 rows (21,252 positive), 32.9s
  trained study model without family 2 (fake_breakout_like), seed 33: best iteration 798, fit 172,888 rows (21,252 positive), 34.1s
  trained study model without family 3 (abnormal_volume_burst), seed 11: best iteration 799, fit 174,598 rows (22,962 positive)

## 3 · Evaluation helpers and tests

In [ ]:
def event_hits(alarm, ev):
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def rolling_inf(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
class Scorer:
    """T1c alarms of a score function fn(sym, key) -> float64 series. q from the genuine DEV bars at 2 episodes/day, per symbol."""
    def __init__(self, fn, bud=PB, qfix=None):
        self.fn = fn; self.q = {}; self.thr = {}
        for s in SYMBOLS:
            g = fn(s, 'GEN_L1'); self.q[s] = B.t1_q(g, L1V, bud) if qfix is None else qfix[s]; self.thr[(s, 'L1')] = rolling_inf(g, self.q[s]); self.thr[(s, 'AO')] = rolling_inf(fn(s, 'GEN_AO'), self.q[s])
    def event_alarms(self, name):
        ref = 'L1' if name == 'VAL' else 'AO'; return [self.fn(s, k) > self.thr[(s, ref)] for s, k, ev in ES[name]]
    def genuine(self, where):
        return {s: (self.fn(s, 'GEN_L1') > self.thr[(s, 'L1')]) if where == 'DEV' else (self.fn(s, 'GEN_AO') > self.thr[(s, 'AO')]) for s in SYMBOLS}
def hits_of(alarms, name): return np.concatenate([event_hits(a, ev) for a, (s, k, ev) in zip(alarms, ES[name])])
def eps_of(alarm_by_sym, where): valid = L1V if where == 'DEV' else CALV; return float(np.mean([B.eps_per_day(alarm_by_sym[s], valid) for s in SYMBOLS]))
def fam_rate(h, name, fam):
    m = META[name]; return float(np.mean([h[((m.sym == s) & (m.fam == fam)).to_numpy()].mean() for s in SYMBOLS]))
def pair_boot(hA, hB, name, fams, n=None, seed=0):
    n = n or NBOOT; m = META[name]; groups = []
    for s in SYMBOLS:
        gs = [np.flatnonzero(((m.sym == s) & (m.fam == f)).to_numpy()) for f in fams]; groups.append([g for g in gs if len(g)])
    d = hA.astype(float) - hB.astype(float); rg = np.random.default_rng(seed); out = []
    for _ in range(n): out.append(np.mean([np.mean([d[g[rg.integers(0, len(g), len(g))]].mean() for g in gs]) for gs in groups]))
    return tuple(float(x) for x in np.percentile(out, [2.5, 97.5]))
# ---- unit tests of the helpers (do not need the study models)
rg = np.random.default_rng(0); _, _, ev_ = ES['CAL'][0]
for _ in range(3):
    al = rg.random(AOLEN) < 0.002; assert (event_hits(al, ev_) == np.array([al[r.t0:r.t0 + r.dur].any() for r in ev_.itertuples()])).all()
say('PASS T02 event hit rule equals a plain loop')
hh = rg.random(len(META['CAL'])) < 0.5; lo_, hi_ = pair_boot(hh, hh, 'CAL', OLD, n=50); assert lo_ == 0 and hi_ == 0; say('PASS T03 paired bootstrap of identical arms is exactly 0')
AEs = Scorer(lambda s, k: AE[(s, k)], qfix=DAE['q_T1_by_budget']['2'])
RQ = {s: B.t1_q(AE[(s, 'GEN_L1')], L1V, PB) for s in SYMBOLS}
say(('PASS' if all(abs(RQ[s] - AEs.q[s]) < 1e-12 for s in SYMBOLS) else 'INFO') + ' T04 AE alarm lines: declared q (used) ' + str(AEs.q) + '; recomputed from the saved scores ' + str(RQ) + ('' if all(abs(RQ[s] - AEs.q[s]) < 1e-12 for s in SYMBOLS) else ' (saved scores are float32, borderline episode counts can move one grid step; the declared q is the deployed one)'))

PASS T02 event hit rule equals a plain loop
PASS T03 paired bootstrap of identical arms is exactly 0
PASS T04 AE alarm lines: declared q (used) {'BTCUSDT': 0.993, 'ETHUSDT': 0.99}; recomputed from the saved scores {'BTCUSDT': 0.993, 'ETHUSDT': 0.99}


## 4 · Results (protocol U1 v2, Part B)

In [ ]:
RESULT = {}
if not TRAIN_DONE: say('study models incomplete -> run all again; nothing evaluated yet')
else:
    # ---- parity of the evaluation with notebook 29b (frozen full XGBoost, VAL-3, T1c, 2/day): only possible when the frozen selection is the studied config
    FROZ_PATHS = [XG/f'N1_A_pooled_pool_d3_s{sd}.json' for sd in SEEDS]; HAVE_FROZ = all(p.exists() and Path(str(p)[:-5] + '.info.json').exists() for p in FROZ_PATHS)
    FZ = SM(FROZ_PATHS) if HAVE_FROZ else None
    if HAVE_FROZ and FROZEN == 'A|pooled|d3':
        fzs = Scorer(FZ.score); h_ = hits_of(fzs.event_alarms('VAL'), 'VAL'); m_ = META['VAL']
        mine = float(np.mean([np.mean([h_[((m_.sym == s) & (m_.fam == f)).to_numpy()].mean() for f in OLD]) for s in SYMBOLS]))
        ref = pd.read_csv(OUT/'xgb_val_table_T1c.csv'); ref = ref[(ref.norm == 'N1') & (ref.model == 'XGBoost v2') & (ref.thr == 'T1c') & (ref['budget/day'] == PB)]['recall 6 old'].iloc[0]
        assert abs(mine - ref) < 1e-9, (mine, ref); say(f'PASS T05 evaluation helpers reproduce notebook 29b: frozen XGBoost v2 VAL-3 recall (6 old, T1c, 2/day) {mine:.4f} = {ref:.4f}')
    else: say('NOTE T05 skipped: frozen selection is not A|pooled|d3 here' + (' or frozen files missing' if not HAVE_FROZ else ''))
    # ---- parity of a re-trained seed-11 study model with the LOFO model notebook 29b saved (informational: thread count can change tree details)
    pr = []
    for fam in OLD:
        old_ = XG/f'N1_A_pooled_pool_d3_s11_no{fam}.json'
        if old_.exists() and Path(str(old_)[:-5] + '.info.json').exists():
            a_, b_ = SM([old_]), SM([spath(fam, 11)]); d_ = float(np.nanmean(np.abs(a_.score(SYMBOLS[0], 'VAL_r0') - b_.score(SYMBOLS[0], 'VAL_r0')))); pr.append((fam, a_.its[0], b_.its[0], d_))
    if pr: say('INFO T06 re-trained seed-11 study models versus notebook 29b LOFO models (family, best iteration old/new, mean |margin difference|): ' + '; '.join(f'{f}: {i0}/{i1}, {d:.4f}' for f, i0, i1, d in pr) + ('  -> identical recipe reproduced' if all(i0 == i1 and d < 1e-3 for f, i0, i1, d in pr) else '  -> small differences (thread count / library version), not a protocol deviation'))
    # ---- the study
    ROWS = []; HITS = {n: {k: np.zeros(len(META[n]), bool) for k in ['S', 'N', 'OR']} for n in META}; GEN = {}
    for fam in OLD:
        sm = SM([spath(fam, sd) for sd in SEEDS]); Ss = Scorer(sm.score); sel = {n: (META[n].fam == fam).to_numpy() for n in META}
        for name in ['VAL', 'CAL']:
            aS, aN = Ss.event_alarms(name), AEs.event_alarms(name); hS, hN = hits_of(aS, name), hits_of(aN, name); hO = hits_of([x | y for x, y in zip(aS, aN)], name)
            assert (hO >= hS).all() and (hO >= hN).all()
            for k, h in [('S', hS), ('N', hN), ('OR', hO)]: HITS[name][k][sel[name]] = h[sel[name]]
        for where in ['DEV', 'CAL']:
            gS, gN = Ss.genuine(where), AEs.genuine(where); GEN[(fam, where)] = {'S': eps_of(gS, where), 'N': eps_of(gN, where), 'OR': eps_of({s: gS[s] | gN[s] for s in SYMBOLS}, where)}
            assert GEN[(fam, where)]['OR'] >= max(GEN[(fam, where)]['S'], GEN[(fam, where)]['N']) - 1e-9
        say(f'  evaluated family {fam} ({B.FAMS[fam]})')
    FZH = {}
    if HAVE_FROZ:
        fzs = Scorer(FZ.score)
        for name in ['VAL', 'CAL']: FZH[name] = hits_of(fzs.event_alarms(name), name)
    say('PASS T07 "S_wo or N" never detects less than either arm, and its genuine burden is at least that of each arm')
    # ---- tables
    for name in ['VAL', 'CAL']:
        for fam in OLD:
            r_ = {'set': name, 'family': B.FAMS[fam], 'events': int((META[name].fam == fam).sum()), 'S_wo (XGB without it)': fam_rate(HITS[name]['S'], name, fam), 'N (AE A6)': fam_rate(HITS[name]['N'], name, fam), 'S_wo or N': fam_rate(HITS[name]['OR'], name, fam)}
            r_['gain'] = r_['S_wo or N'] - r_['S_wo (XGB without it)']
            sel_ = (META[name].fam == fam).to_numpy(); r_['events AE rescued'] = int((HITS[name]['OR'][sel_] & ~HITS[name]['S'][sel_]).sum())
            if name in FZH: r_['XGB trained on it'] = fam_rate(FZH[name], name, fam)
            ROWS.append(r_)
    FT = pd.DataFrame(ROWS); FT.to_csv(OUT/'u1_partB_family_table.csv', index=False)
    say('--- per family, 2 episodes/day (mean BTC/ETH). VAL-3 = design, CAL-3 = decision ---'); say(FT.round(3).to_string(index=False))
    def macro(name, k): return float(np.mean([np.mean([HITS[name][k][((META[name].sym == s) & (META[name].fam == f)).to_numpy()].mean() for f in OLD]) for s in SYMBOLS]))
    SUMM = {}
    for name in ['VAL', 'CAL']:
        lo, hi = pair_boot(HITS[name]['OR'], HITS[name]['S'], name, OLD); SUMM[name] = {'S_wo': macro(name, 'S'), 'N': macro(name, 'N'), 'OR': macro(name, 'OR'), 'gain': macro(name, 'OR') - macro(name, 'S'), 'gain_ci': [lo, hi]}
        if name in FZH: SUMM[name]['XGB_trained_on_it'] = float(np.mean([np.mean([FZH[name][((META[name].sym == s) & (META[name].fam == f)).to_numpy()].mean() for f in OLD]) for s in SYMBOLS]))
    say('--- macro over the six families ---')
    for name in SUMM: say(f'{name}: XGB without the family {SUMM[name]["S_wo"]:.3f}, AE {SUMM[name]["N"]:.3f}, either {SUMM[name]["OR"]:.3f}; gain of adding the AE {SUMM[name]["gain"]:+.3f} (paired 95% {SUMM[name]["gain_ci"][0]:+.3f}..{SUMM[name]["gain_ci"][1]:+.3f})' + (f'; XGB trained on the family {SUMM[name]["XGB_trained_on_it"]:.3f}' if 'XGB_trained_on_it' in SUMM[name] else ''))
    BT = pd.DataFrame([{'family': B.FAMS[f], 'where': w, 'S_wo eps/day': GEN[(f, w)]['S'], 'AE eps/day': GEN[(f, w)]['N'], 'S_wo or N eps/day': GEN[(f, w)]['OR'], 'added by AE': GEN[(f, w)]['OR'] - GEN[(f, w)]['S']} for f in OLD for w in ['DEV', 'CAL']]); BT.to_csv(OUT/'u1_partB_burden_table.csv', index=False)
    say('--- genuine burden (episodes/day, mean BTC/ETH): DEV = Layer 1 DEV bars, CAL = genuine August ---'); say(BT.round(3).to_string(index=False))
    cal_S = float(np.mean([GEN[(f, 'CAL')]['S'] for f in OLD])); cal_O = float(np.mean([GEN[(f, 'CAL')]['OR'] for f in OLD])); worst = max(GEN[(f, 'CAL')]['OR'] - GEN[(f, 'CAL')]['S'] for f in OLD)
    hu4 = bool(SUMM['CAL']['gain'] >= 0.05 and SUMM['CAL']['gain_ci'][0] > 0); hu5 = bool(cal_O <= cal_S + 2.1)
    RESULT = {'protocol_u1_v2_sha256': PHU, 'partB_readings_sha256': RH, 'summary': SUMM, 'H-U4': {'value': SUMM['CAL']['gain'], 'ci': SUMM['CAL']['gain_ci'], 'holds': hu4}, 'H-U5': {'S_wo_mean_eps_day': cal_S, 'OR_mean_eps_day': cal_O, 'worst_family_addition': worst, 'holds': hu5}, 'mock': MOCK}
    say('')
    say('=== pre-registered hypotheses (protocol U1 v2, Part B; decided on CAL-3) ===')
    say(f'H-U4 (the AE adds on a truly unseen family: macro gain >= 0.05 and interval above 0): gain {SUMM["CAL"]["gain"]:+.3f}, 95% {SUMM["CAL"]["gain_ci"][0]:+.3f}..{SUMM["CAL"]["gain_ci"][1]:+.3f} -> {"HOLDS" if hu4 else "FAILS"}')
    say(f'H-U5 (burden: "S_wo or N" <= S_wo + 2.1 episodes/day on CAL-3 genuine): {cal_O:.2f} vs {cal_S:.2f} + 2.1 = {cal_S + 2.1:.2f}; worst single family adds {worst:.2f} -> {"HOLDS" if hu5 else "FAILS"}')
    say(('Both hold: the AE is evidence for previously unseen behaviour on this synthetic test; Part C is not needed.' if hu4 and hu5 else 'H-U4 failed: Part C (AE v3 as a new version) is opened as a separate protocol; nothing here is retuned.' if not hu4 else 'H-U4 holds but H-U5 failed: the AE helps but costs more alarms than declared; reported as failed, nothing retuned.'))
    say('Failed hypotheses are reported as failed. Labels are synthetic; the AE variant A6 was selected on labelled VAL-3, so it is not purely label-free.')

PASS T05 evaluation helpers reproduce notebook 29b: frozen XGBoost v2 VAL-3 recall (6 old, T1c, 2/day) 0.6184 = 0.6184
INFO T06 re-trained seed-11 study models versus notebook 29b LOFO models (family, best iteration old/new, mean |margin difference|): 1: 796/796, 0.0000; 2: 799/799, 0.0000; 3: 799/799, 0.0000; 4: 799/799, 0.0000; 5: 798/798, 0.0000; 6: 796/796, 0.0000  -> identical recipe reproduced
  evaluated family 1 (pump_and_dump_like)
  evaluated family 2 (fake_breakout_like)
  evaluated family 3 (abnormal_volume_burst)
  evaluated family 4 (trade_flow_imbalance)
  evaluated family 5 (wash_like_aggregate_proxy)
  evaluated family 6 (liquidity_stress_proxy)
PASS T07 "S_wo or N" never detects less than either arm, and its genuine burden is at least that of each arm
--- per family, 2 episodes/day (mean BTC/ETH). VAL-3 = design, CAL-3 = decision ---
set                    family  events  S_wo (XGB without it)  N (AE A6)  S_wo or N  gain  events AE rescued  XGB trained on it
VAL      

## 5 · Save

In [ ]:
if RESULT:
    (OUT/'u1_partB_results.json').write_text(json.dumps(RESULT, indent=1, default=str)); (OUT/'u1_partB_share_block.txt').write_text('\n'.join(LINES))
    say('Saved: u1_partB_results.json, u1_partB_family_table.csv, u1_partB_burden_table.csv, u1_partB_share_block.txt in models_b2_v1/ . Send the share block to Claude.')
else: say('Nothing saved yet (not all study models exist). Run all again.')

Saved: u1_partB_results.json, u1_partB_family_table.csv, u1_partB_burden_table.csv, u1_partB_share_block.txt in models_b2_v1/ . Send the share block to Claude.
